

## Setup

In [ ]:
%%capture

%pip install "git+https://github.com/somombo/impalab.git@python-sdk#subdirectory=python-sdk"

In [ ]:
import os
os.environ['RUST_LOG'] = "debug"
os.environ['NO_COLOR'] = "1"

In [ ]:
from impalab_py.colab import colab_repo_sync

colab_repo_sync(
  repo_url="https://github.com/somombo/sort-bench.git",
  repo_subdir="lab",
)


## Experiment Config

In [ ]:
from impalab_py import Impa
import json

impa = Impa(
  root_dir="..",
)


In [ ]:
from impalab_py.colab import is_colab_env
import os

if is_colab_env() or (os.getenv("GITHUB_ACTIONS") == "true"): 
  impa.download_executable()

  from env_setup import *

  setup_lean()

Let us run a warmup round of the benchmark to pre-build build all the executables

In [ ]:
_build_result = impa.build(
  components_dir="./components",
  include=[
    'somombo_unifshuffle_multi',
    'c',
    'python',
  ]
)
assert _build_result, "Initial build failed"

In [ ]:
_test_results = impa.run(
  generator={
    "name": "somombo_unifshuffle_multi",
    "seed": 42,
    "args": [json.dumps([
      {
        "cardinality": 19,
        "multiplicity": 13,
        "swaps": 2,
        "descending": True,
        "runs": 7,
      }, 
      {
        "cardinality": 17,
        "multiplicity": 11,
        "swaps": None,
        "descending": False,
        "runs": 5,
      }
    ])],
  }, 
  reps=3, 
  tasks=[
    {'executor': 'c', 'args': ['qsort']},
    {'executor': 'python', 'args': ['list.sort']},
  ],
  attributes={
    "experiment_name": "Warmup-Experiment",
  },
)

In [ ]:
from exploration import ExplorerFromLabDf

ExplorerFromLabDf(_test_results).get_clean_data()